# 14 — Pipeline v3: Ajuste Transdutivo + Features Adicionais Removidas
## PRT Seguros

Este notebook refaz o pipeline de preparação (`00`) do zero, incorporando tudo que aprendemos nos
notebooks `07`-`13`:

1. **Mesma correção do bug de região** (`regiao_Centro-Oeste`)
2. **Remove as 4 features numéricas fracas+shift** encontradas no notebook `09`
   (`valor_premio_anual`, `km_anual_estimado`, `tempo_medio_resposta_dias`, `ano_veiculo`)
3. **Remove as 3 colunas categóricas fracas+shift** encontradas no notebook `13`
   (`canal_Nao Informado`, `veiculo_NaN`, `segmento_NaN`)
4. **Ajuste transdutivo**: o `SimpleImputer`, o `StandardScaler` e o `KMeans` são ajustados usando a
   **união das features de treino + Kaggle** (nunca a validação, e nunca o `churned`) — a ideia
   validada com você: isso não ensina nada de novo sobre churn, só faz a "régua de medição"
   (mediana de imputação, escala, centróides de cluster) refletir a população real, não só a
   fatia de treino.
5. **Mesmas features derivadas** do notebook `10` (razões e interações)

No final, treinamos o CatBoost com os hiperparâmetros tunados do notebook `11` sobre esse novo
conjunto (`_v3`) e comparamos com o resultado do `_v2` (AUC-ROC validação = 0.8263, Kaggle público = 0.7370).


## 1. Imports e carregar bases brutas

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

df_train = pd.read_csv("../bases/tabelas_unificadas/Base_Unificada_Outer.csv")
df_kaggle = pd.read_csv("../bases/bases_kaggle/Base_Unificada_Kaggle_Outer.csv")
print(f"Treino: {df_train.shape} | Kaggle: {df_kaggle.shape}")


Treino: (100000, 84) | Kaggle: (100000, 77)


## 2. Alinhar colunas e corrigir o bug de região (igual ao notebook `00`)

In [2]:
train_cols_lower = {c.lower(): c for c in df_train.columns}
rename_map = {
    c: train_cols_lower[c.lower()]
    for c in df_kaggle.columns
    if c not in df_train.columns and c.lower() in train_cols_lower
}
df_kaggle = df_kaggle.rename(columns=rename_map)

REGIAO_CENTRO_OESTE_VARIANTES = ["regiao_Centro", "regiao_Oeste", "regiao_Regiao Oeste"]
df_train["regiao_Centro-Oeste"] = df_train[
    ["regiao_Centro-Oeste"] + REGIAO_CENTRO_OESTE_VARIANTES
].max(axis=1)

DROP_COLS = [
    "score_propensao_churn", "cluster_sugerido_crm",
    "data_primeira_apolice", "data_nascimento",
    "renovacoes_consecutivas",
    "num_apolices_basica", "num_apolices_padrao", "num_apolices_premium",
    "regiao_Centro", "regiao_Oeste", "regiao_Regiao Oeste",
    # novidades desta versão v3:
    "valor_premio_anual", "km_anual_estimado", "tempo_medio_resposta_dias", "ano_veiculo",
    "canal_Nao Informado", "veiculo_NaN", "segmento_NaN",
]

df_train = df_train.drop(columns=[c for c in DROP_COLS if c in df_train.columns])
df_kaggle = df_kaggle.drop(columns=[c for c in DROP_COLS if c in df_kaggle.columns])

feat_train = set(df_train.columns) - {TARGET_COL}
feat_kaggle = set(df_kaggle.columns)
assert feat_train == feat_kaggle, f"Colunas divergentes: {feat_train ^ feat_kaggle}"
print(f"OK — {len(feat_train)} features idênticas em treino e Kaggle (v3, já com as remoções extras).")


OK — 65 features idênticas em treino e Kaggle (v3, já com as remoções extras).


## 3. Split treino/validação (mesmo `random_state`, para os resultados ficarem comparáveis)

In [3]:
df_train_split, df_val_split = train_test_split(
    df_train, test_size=0.2, stratify=df_train[TARGET_COL], random_state=RANDOM_STATE
)
df_train_split = df_train_split.reset_index(drop=True)
df_val_split = df_val_split.reset_index(drop=True)
print(f"Treino: {df_train_split.shape} | Val: {df_val_split.shape}")


Treino: (80000, 66) | Val: (20000, 66)


## 4. Ajuste TRANSDUTIVO: cluster ajustado com treino + Kaggle (sem o rótulo)

Diferença-chave em relação ao notebook `00`: em vez de `imp_cluster.fit(df_train_split[...])`,
fazemos `fit` na **união** de `df_train_split` e `df_kaggle` (concatenando só as features, nunca o
`churned`, que nem existe no `df_kaggle`). A validação continua de fora do `fit` — ela representa
dados que o pipeline nunca viu, para mantermos uma métrica honesta.

In [4]:
FEATURES_CLUSTER = [
    "tempo_cliente_dias", "num_apolices_ativas", "num_produtos_contratados",
    "desconto_aplicado_pct", "indice_relacionamento", "satisfacao_nps",
    "tipo_cobertura_basica", "estado_civil", "tipo_cobertura_premium",
    "tem_filhos", "renda_anual", "qtd_dependentes", "valor_imovel",
    "valor_cobertura_total", "idade", "score_engajamento_digital",
    "pagamento_em_dia", "num_reclamacoes_12m", "num_acessos_app_mes",
    "possui_imovel", "tipo_cobertura_padrao",
]  # sem valor_premio_anual e franquia_media/km_anual_estimado etc. que já saíram do dataset
FEATURES_CLUSTER = [c for c in FEATURES_CLUSTER if c in df_train_split.columns]
K_FINAL = 6

pool_fit_cluster = pd.concat(
    [df_train_split[FEATURES_CLUSTER], df_kaggle[FEATURES_CLUSTER]], ignore_index=True
)

imp_cluster = SimpleImputer(strategy="median").fit(pool_fit_cluster)
scaler_cluster = StandardScaler().fit(imp_cluster.transform(pool_fit_cluster))

def montar_cluster(df):
    return scaler_cluster.transform(imp_cluster.transform(df[FEATURES_CLUSTER]))

kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10).fit(
    montar_cluster(pool_fit_cluster)
)

df_train_split["cluster"] = kmeans.predict(montar_cluster(df_train_split))
df_val_split["cluster"] = kmeans.predict(montar_cluster(df_val_split))
df_kaggle["cluster"] = kmeans.predict(montar_cluster(df_kaggle))

print("Distribuição de cluster — treino:")
print(df_train_split["cluster"].value_counts(normalize=True).sort_index().round(3))
print("Distribuição de cluster — Kaggle:")
print(df_kaggle["cluster"].value_counts(normalize=True).sort_index().round(3))


Distribuição de cluster — treino:
cluster
0    0.056
1    0.224
2    0.192
3    0.237
4    0.006
5    0.284
Name: proportion, dtype: float64
Distribuição de cluster — Kaggle:
cluster
0    0.055
1    0.226
2    0.193
3    0.233
4    0.007
5    0.286
Name: proportion, dtype: float64


## 5. Imputação final — também transdutiva (treino + Kaggle, sem validação nem rótulo)

In [5]:
feature_cols = [c for c in df_train_split.columns if c not in (ID_COL, TARGET_COL)]

pool_fit_final = pd.concat(
    [df_train_split[feature_cols], df_kaggle[feature_cols]], ignore_index=True
)
imputer_final = SimpleImputer(strategy="median").fit(pool_fit_final)

def montar_final(df, tem_target):
    saida = pd.DataFrame(imputer_final.transform(df[feature_cols]), columns=feature_cols)
    saida[ID_COL] = df[ID_COL].values
    if tem_target:
        saida[TARGET_COL] = df[TARGET_COL].values
    return saida

train_v3 = montar_final(df_train_split, tem_target=True)
val_v3 = montar_final(df_val_split, tem_target=True)
kaggle_v3 = montar_final(df_kaggle, tem_target=False)

assert train_v3.isna().sum().sum() == 0
assert val_v3.isna().sum().sum() == 0
assert kaggle_v3.isna().sum().sum() == 0
print(f"train_v3 : {train_v3.shape}")
print(f"val_v3   : {val_v3.shape}")
print(f"kaggle_v3: {kaggle_v3.shape}")


train_v3 : (80000, 67)
val_v3   : (20000, 67)
kaggle_v3: (100000, 66)


## 6. Features derivadas (mesmas do notebook `10`, exceto as 2 que dependiam de
`valor_premio_anual` — removida do v3 no passo 2)

In [6]:
def criar_features(df):
    df = df.copy()
    df["renda_por_dependente"] = df["renda_anual"] / (df["qtd_dependentes"] + 1)
    df["valor_imovel_por_renda"] = df["valor_imovel"] / (df["renda_anual"] + 1)
    df["apolices_por_produto"] = df["num_apolices_ativas"] / (df["num_produtos_contratados"] + 1)
    df["reclamacoes_por_sinistro"] = df["num_reclamacoes_12m"] / (df["num_sinistros_historico"] + 1)
    df["ligacoes_por_sinistro"] = df["num_ligacoes_suporte_12m"] / (df["num_sinistros_historico"] + 1)
    df["engajamento_x_satisfacao"] = df["score_engajamento_digital"] * df["satisfacao_nps"]
    df["desconto_x_tempo_cliente"] = df["desconto_aplicado_pct"] * df["tempo_cliente_dias"]
    df["relacionamento_por_ano_cliente"] = df["indice_relacionamento"] / (df["tempo_cliente_dias"] / 365 + 1)
    df["contato_menos_login"] = df["dias_ultimo_contato"] - df["ultimo_login_portal_dias"]
    return df

train_v3 = criar_features(train_v3)
val_v3 = criar_features(val_v3)
kaggle_v3 = criar_features(kaggle_v3)

feature_cols_v3 = [c for c in train_v3.columns if c not in (ID_COL, TARGET_COL)]
print(f"Total de features v3: {len(feature_cols_v3)}")


Total de features v3: 74


## 7. Salvar bases `_v3`

In [7]:
train_v3.to_csv("dados_processados/train_model_ready_v3.csv", index=False)
val_v3.to_csv("dados_processados/val_model_ready_v3.csv", index=False)
kaggle_v3.to_csv("dados_processados/kaggle_model_ready_v3.csv", index=False)
print("Arquivos _v3 salvos em dados_processados/")


Arquivos _v3 salvos em dados_processados/


## 8. Treinar CatBoost (hiperparâmetros tunados do notebook `11`) e comparar com o `_v2`

In [8]:
X_train, y_train = train_v3[feature_cols_v3], train_v3[TARGET_COL]
X_val, y_val = val_v3[feature_cols_v3], val_v3[TARGET_COL]
X_kaggle = kaggle_v3[feature_cols_v3]

X_tr_es, X_es, y_tr_es, y_es = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
neg_pos_ratio_es = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}

cat_v3 = CatBoostClassifier(
    iterations=3000, **MELHORES_PARAMS_CAT,
    scale_pos_weight=neg_pos_ratio_es, eval_metric="AUC",
    random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
)
cat_v3.fit(X_tr_es, y_tr_es, eval_set=(X_es, y_es))

proba_val_v3 = cat_v3.predict_proba(X_val)[:, 1]
auc_v3 = roc_auc_score(y_val, proba_val_v3)

print(f"CatBoost tuned + v2 (sem ajuste transdutivo): AUC-ROC val = 0.8263 | Kaggle público = 0.7370")
print(f"CatBoost tuned + v3 (com ajuste transdutivo) : AUC-ROC val = {auc_v3:.4f}")


CatBoost tuned + v2 (sem ajuste transdutivo): AUC-ROC val = 0.8263 | Kaggle público = 0.7370
CatBoost tuned + v3 (com ajuste transdutivo) : AUC-ROC val = 0.8261


## 9. Gerar submissão Kaggle do modelo v3

In [9]:
proba_kaggle_v3 = cat_v3.predict_proba(X_kaggle)[:, 1]
submissao_v3 = pd.DataFrame({"Id": kaggle_v3[ID_COL], "probabilidade_churn": proba_kaggle_v3})
submissao_v3.to_csv("submissions/submission_catboost_v3_transdutivo.csv", index=False)
print("Salvo: submissions/submission_catboost_v3_transdutivo.csv")
submissao_v3.head()


Salvo: submissions/submission_catboost_v3_transdutivo.csv


,Id,probabilidade_churn
0,221300000002,0.092628
1,221300000020,0.276877
2,221300000097,0.164887
3,221300000148,0.436827
4,221300000166,0.314010
